# Decision Trees - Comprehensive Guide with Hyperparameter Tuning

This notebook provides an in-depth exploration of Decision Trees for network intrusion detection, with a focus on:
- Understanding each hyperparameter
- Practical examples of tuning
- Visualization of tree behavior
- Performance optimization strategies

## 1. Data Loading and Exploration

In [ ]:
# Load data
print("Loading dataset...")
df = pd.read_csv('data/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check dataset info
print("Dataset Information:")
df.info()

In [ ]:
# Check for label column
print("Column names:")
print(df.columns.tolist())

In [ ]:
# Find the label column (usually named 'Label' or similar)
label_col = [col for col in df.columns if 'label' in col.lower()]
if label_col:
    label_col = label_col[0]
    print(f"Label column found: {label_col}")
    print(f"\nUnique labels: {df[label_col].unique()}")
    print(f"\nLabel distribution:")
    print(df[label_col].value_counts())
else:
    print("No label column found. Last column will be used as label.")
    label_col = df.columns[-1]
    print(f"Using: {label_col}")

## 2. Data Preprocessing

In [ ]:
# Check for missing values
print("Missing values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("No missing values found!")

# Check for infinite values
print("\nChecking for infinite values...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
if df.isnull().sum().sum() > 0:
    print(f"Found {df.isnull().sum().sum()} infinite/NaN values. Filling with 0...")
    df.fillna(0, inplace=True)
else:
    print("No infinite values found!")

In [ ]:
# Separate features and labels
X = df.drop(columns=[label_col])
y = df[label_col]

# Encode labels if they're strings
if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)
    label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print("Label encoding:")
    for label, code in label_mapping.items():
        print(f"  {label}: {code}")
else:
    label_mapping = None

print(f"\nFeature shape: {X.shape}")
print(f"Label shape: {y.shape}")
print(f"Number of classes: {len(np.unique(y))}")

In [ ]:
# Handle non-numeric features
print("Non-numeric columns:")
non_numeric = X.select_dtypes(include=['object']).columns
if len(non_numeric) > 0:
    print(non_numeric.tolist())
    print("\nDropping non-numeric columns...")
    X = X.select_dtypes(exclude=['object'])
else:
    print("All features are numeric!")

print(f"\nFinal feature count: {X.shape[1]}")

In [ ]:
# For faster training, let's sample the data if it's too large
if len(df) > 100000:
    print(f"Dataset is large ({len(df)} rows). Sampling 100,000 rows for faster training...")
    sample_indices = np.random.choice(len(df), 100000, replace=False)
    X = X.iloc[sample_indices]
    y = y[sample_indices]
    print(f"New shape: {X.shape}")
else:
    print(f"Using full dataset: {X.shape}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution in training set:")
unique, counts = np.unique(y_train, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count/len(y_train)*100:.2f}%)")

## 3. Baseline Decision Tree

Let's start with a default Decision Tree to establish a baseline.

In [ ]:
# Train baseline model
print("Training baseline Decision Tree...")
start_time = time.time()

baseline_dt = DecisionTreeClassifier(random_state=42)
baseline_dt.fit(X_train, y_train)

train_time = time.time() - start_time
print(f"Training completed in {train_time:.2f} seconds")

# Predictions
y_pred = baseline_dt.predict(X_test)

# Evaluate
print(f"\nBaseline Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"\nTree depth: {baseline_dt.get_depth()}")
print(f"Number of leaves: {baseline_dt.get_n_leaves()}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Baseline Decision Tree - Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 4. Decision Tree Hyperparameters - Deep Dive

### 4.1 max_depth: Maximum Depth of the Tree

**What it does:** Limits how deep the tree can grow.

**Impact:**
- **Low values (1-5):** Underfitting, simple model, fast training
- **Medium values (10-20):** Balanced complexity
- **High values (>30) or None:** Overfitting, complex model, slow training

**When to adjust:**
- Decrease if model is overfitting (high training accuracy, low test accuracy)
- Increase if model is underfitting (low accuracy on both training and test)

In [ ]:
# Experiment with max_depth
print("Testing different max_depth values...\n")

max_depths = [3, 5, 10, 15, 20, 25, None]
results_depth = []

for depth in max_depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
    # Train
    start = time.time()
    dt.fit(X_train, y_train)
    train_time = time.time() - start
    
    # Evaluate
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    
    results_depth.append({
        'max_depth': depth if depth else 'None',
        'actual_depth': dt.get_depth(),
        'n_leaves': dt.get_n_leaves(),
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'overfit_gap': train_acc - test_acc,
        'train_time': train_time
    })
    
    print(f"max_depth={depth}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, "
          f"Depth={dt.get_depth()}, Time={train_time:.3f}s")

results_depth_df = pd.DataFrame(results_depth)
print("\nSummary:")
print(results_depth_df)

In [ ]:
# Visualize max_depth impact
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Accuracy comparison
x_pos = range(len(results_depth_df))
axes[0].plot(x_pos, results_depth_df['train_accuracy'], marker='o', label='Train Accuracy', linewidth=2)
axes[0].plot(x_pos, results_depth_df['test_accuracy'], marker='s', label='Test Accuracy', linewidth=2)
axes[0].set_xlabel('max_depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Training vs Test Accuracy')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(results_depth_df['max_depth'])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Overfitting gap
axes[1].bar(x_pos, results_depth_df['overfit_gap'], color='coral', alpha=0.7)
axes[1].set_xlabel('max_depth')
axes[1].set_ylabel('Overfitting Gap')
axes[1].set_title('Overfitting (Train Acc - Test Acc)')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(results_depth_df['max_depth'])
axes[1].grid(True, alpha=0.3)

# Plot 3: Training time
axes[2].bar(x_pos, results_depth_df['train_time'], color='steelblue', alpha=0.7)
axes[2].set_xlabel('max_depth')
axes[2].set_ylabel('Training Time (seconds)')
axes[2].set_title('Training Time vs max_depth')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(results_depth_df['max_depth'])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- As max_depth increases, training accuracy improves but test accuracy may plateau or decrease")
print("- Large overfitting gap indicates the model is memorizing training data")
print("- Training time increases with depth due to more complex trees")

### 4.2 min_samples_split: Minimum Samples to Split a Node

**What it does:** Minimum number of samples required to split an internal node.

**Impact:**
- **Low values (2-5):** More splits, deeper trees, possible overfitting
- **High values (50-200):** Fewer splits, shallower trees, better generalization

**When to adjust:**
- Increase to prevent overfitting
- Decrease if model is too simple and underfitting

In [ ]:
# Experiment with min_samples_split
print("Testing different min_samples_split values...\n")

min_splits = [2, 10, 50, 100, 200, 500]
results_split = []

for split in min_splits:
    dt = DecisionTreeClassifier(min_samples_split=split, random_state=42)
    
    start = time.time()
    dt.fit(X_train, y_train)
    train_time = time.time() - start
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    
    results_split.append({
        'min_samples_split': split,
        'depth': dt.get_depth(),
        'n_leaves': dt.get_n_leaves(),
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'overfit_gap': train_acc - test_acc,
        'train_time': train_time
    })
    
    print(f"min_samples_split={split}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, "
          f"Depth={dt.get_depth()}, Leaves={dt.get_n_leaves()}")

results_split_df = pd.DataFrame(results_split)

In [ ]:
# Visualize min_samples_split impact
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Accuracy
axes[0, 0].plot(results_split_df['min_samples_split'], results_split_df['train_accuracy'], 
                marker='o', label='Train', linewidth=2)
axes[0, 0].plot(results_split_df['min_samples_split'], results_split_df['test_accuracy'], 
                marker='s', label='Test', linewidth=2)
axes[0, 0].set_xlabel('min_samples_split')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Accuracy vs min_samples_split')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Tree complexity
axes[0, 1].plot(results_split_df['min_samples_split'], results_split_df['depth'], 
                marker='o', label='Depth', linewidth=2)
axes[0, 1].plot(results_split_df['min_samples_split'], results_split_df['n_leaves']/10, 
                marker='s', label='Leaves/10', linewidth=2)
axes[0, 1].set_xlabel('min_samples_split')
axes[0, 1].set_ylabel('Tree Complexity')
axes[0, 1].set_title('Tree Complexity vs min_samples_split')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Overfitting
axes[1, 0].bar(range(len(results_split_df)), results_split_df['overfit_gap'], 
               color='coral', alpha=0.7)
axes[1, 0].set_xlabel('min_samples_split')
axes[1, 0].set_ylabel('Overfitting Gap')
axes[1, 0].set_title('Overfitting vs min_samples_split')
axes[1, 0].set_xticks(range(len(results_split_df)))
axes[1, 0].set_xticklabels(results_split_df['min_samples_split'])
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Training time
axes[1, 1].bar(range(len(results_split_df)), results_split_df['train_time'], 
               color='steelblue', alpha=0.7)
axes[1, 1].set_xlabel('min_samples_split')
axes[1, 1].set_ylabel('Training Time (s)')
axes[1, 1].set_title('Training Time vs min_samples_split')
axes[1, 1].set_xticks(range(len(results_split_df)))
axes[1, 1].set_xticklabels(results_split_df['min_samples_split'])
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 min_samples_leaf: Minimum Samples in Leaf Nodes

**What it does:** Minimum number of samples required to be at a leaf node.

**Impact:**
- **Low values (1-5):** More leaves, complex model, potential overfitting
- **High values (20-100):** Fewer leaves, simpler model, better generalization

**When to adjust:**
- Increase to smooth out predictions and reduce overfitting
- This is often more effective than min_samples_split for regularization

In [ ]:
# Experiment with min_samples_leaf
print("Testing different min_samples_leaf values...\n")

min_leafs = [1, 5, 10, 20, 50, 100]
results_leaf = []

for leaf in min_leafs:
    dt = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=42)
    
    start = time.time()
    dt.fit(X_train, y_train)
    train_time = time.time() - start
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    
    results_leaf.append({
        'min_samples_leaf': leaf,
        'depth': dt.get_depth(),
        'n_leaves': dt.get_n_leaves(),
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'overfit_gap': train_acc - test_acc,
        'train_time': train_time
    })
    
    print(f"min_samples_leaf={leaf}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}, "
          f"Leaves={dt.get_n_leaves()}")

results_leaf_df = pd.DataFrame(results_leaf)
print("\n", results_leaf_df)

### 4.4 max_features: Maximum Features to Consider per Split

**What it does:** Number of features to consider when looking for the best split.

**Options:**
- **None or 'auto':** Use all features
- **'sqrt':** Use sqrt(n_features)
- **'log2':** Use log2(n_features)
- **int:** Use specific number of features
- **float:** Use percentage of features

**Impact:**
- Limiting features can reduce overfitting and speed up training
- Too few features may miss important patterns

In [ ]:
# Experiment with max_features
print("Testing different max_features values...\n")
print(f"Total features: {X_train.shape[1]}")
print(f"sqrt(features): {int(np.sqrt(X_train.shape[1]))}")
print(f"log2(features): {int(np.log2(X_train.shape[1]))}\n")

max_features_list = [None, 'sqrt', 'log2', 0.5, 10]
results_features = []

for feat in max_features_list:
    dt = DecisionTreeClassifier(max_features=feat, random_state=42)
    
    start = time.time()
    dt.fit(X_train, y_train)
    train_time = time.time() - start
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    
    results_features.append({
        'max_features': str(feat),
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'overfit_gap': train_acc - test_acc,
        'train_time': train_time
    })
    
    print(f"max_features={feat}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")

results_features_df = pd.DataFrame(results_features)

### 4.5 criterion: Split Quality Measure

**What it does:** Function to measure the quality of a split.

**Options:**
- **'gini':** Gini impurity (default) - faster, tends to isolate most frequent class
- **'entropy':** Information gain - may create more balanced trees
- **'log_loss':** Log loss (for probabilistic predictions)

**When to use:**
- Use 'gini' for speed and when classes are imbalanced
- Use 'entropy' when you want more balanced splits

In [ ]:
# Experiment with criterion
print("Testing different criterion values...\n")

criteria = ['gini', 'entropy']
results_criterion = []

for crit in criteria:
    dt = DecisionTreeClassifier(criterion=crit, random_state=42)
    
    start = time.time()
    dt.fit(X_train, y_train)
    train_time = time.time() - start
    
    train_acc = dt.score(X_train, y_train)
    test_acc = dt.score(X_test, y_test)
    y_pred = dt.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results_criterion.append({
        'criterion': crit,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'f1_score': f1,
        'train_time': train_time
    })
    
    print(f"criterion={crit}: Test Acc={test_acc:.4f}, F1={f1:.4f}, Time={train_time:.3f}s")

results_criterion_df = pd.DataFrame(results_criterion)
print("\n", results_criterion_df)

### 4.6 class_weight: Handling Class Imbalance

**What it does:** Weights associated with classes to handle imbalance.

**Options:**
- **None:** All classes have equal weight
- **'balanced':** Automatically adjust weights inversely proportional to class frequencies
- **dict:** Custom weights for each class

**When to use:**
- Use 'balanced' when you have imbalanced classes
- This helps prevent bias toward majority class

In [ ]:
# Check class balance
unique, counts = np.unique(y_train, return_counts=True)
print("Class distribution:")
for label, count in zip(unique, counts):
    print(f"  Class {label}: {count} ({count/len(y_train)*100:.2f}%)")

# Test with and without class weighting
print("\nComparing class_weight settings...\n")

for cw in [None, 'balanced']:
    dt = DecisionTreeClassifier(class_weight=cw, max_depth=10, random_state=42)
    dt.fit(X_train, y_train)
    
    y_pred = dt.predict(X_test)
    
    print(f"class_weight={cw}:")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"  F1-Score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print(f"  F1-Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
    print()

## 5. Grid Search for Optimal Hyperparameters

Now let's use GridSearchCV to find the best combination of hyperparameters.

In [ ]:
# Define parameter grid
param_grid = {
    'max_depth': [10, 15, 20, 25],
    'min_samples_split': [10, 50, 100],
    'min_samples_leaf': [5, 10, 20],
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2', None]
}

print("Starting Grid Search...")
print(f"Total combinations to test: {np.prod([len(v) for v in param_grid.values()])}")
print("This may take a few minutes...\n")

# Perform grid search
dt = DecisionTreeClassifier(random_state=42)
grid_search = GridSearchCV(
    dt, param_grid, cv=3, scoring='accuracy', 
    n_jobs=-1, verbose=1
)

start_time = time.time()
grid_search.fit(X_train, y_train)
search_time = time.time() - start_time

print(f"\nGrid Search completed in {search_time:.2f} seconds")
print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate best model
best_dt = grid_search.best_estimator_
y_pred_best = best_dt.predict(X_test)

print("Best Model Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_best, average='weighted'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_best, average='weighted'):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_best, average='weighted'):.4f}")

print(f"\nTree Properties:")
print(f"Depth: {best_dt.get_depth()}")
print(f"Leaves: {best_dt.get_n_leaves()}")

In [ ]:
# Compare baseline vs optimized
comparison_data = {
    'Model': ['Baseline', 'Optimized'],
    'Accuracy': [
        accuracy_score(y_test, baseline_dt.predict(X_test)),
        accuracy_score(y_test, y_pred_best)
    ],
    'F1-Score': [
        f1_score(y_test, baseline_dt.predict(X_test), average='weighted'),
        f1_score(y_test, y_pred_best, average='weighted')
    ],
    'Tree Depth': [
        baseline_dt.get_depth(),
        best_dt.get_depth()
    ],
    'Num Leaves': [
        baseline_dt.get_n_leaves(),
        best_dt.get_n_leaves()
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nBaseline vs Optimized Comparison:")
print(comparison_df)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Performance metrics
metrics = ['Accuracy', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, comparison_df.iloc[0][1:3], width, label='Baseline', alpha=0.8)
axes[0].bar(x + width/2, comparison_df.iloc[1][1:3], width, label='Optimized', alpha=0.8)
axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Performance Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Tree complexity
complexity = ['Tree Depth', 'Num Leaves']
x = np.arange(len(complexity))

axes[1].bar(x - width/2, comparison_df.iloc[0][3:5], width, label='Baseline', alpha=0.8)
axes[1].bar(x + width/2, comparison_df.iloc[1][3:5], width, label='Optimized', alpha=0.8)
axes[1].set_xlabel('Property')
axes[1].set_ylabel('Value')
axes[1].set_title('Tree Complexity Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(complexity)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Feature Importance Analysis

Decision trees provide feature importance scores that tell us which features are most useful for classification.

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_dt.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance.head(20))

# Visualize top features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Learning Curves

Learning curves help us understand if our model would benefit from more training data.

In [ ]:
# Generate learning curves
print("Generating learning curves (this may take a moment)...")

train_sizes, train_scores, val_scores = learning_curve(
    best_dt, X_train, y_train,
    cv=3, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy'
)

# Calculate mean and std
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Plot learning curves
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, label='Training score', marker='o', linewidth=2)
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
plt.plot(train_sizes, val_mean, label='Validation score', marker='s', linewidth=2)
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy Score')
plt.title('Learning Curves - Optimized Decision Tree')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- If training and validation curves converge, model is well-tuned")
print("- Large gap indicates overfitting")
print("- Both curves still improving means more data could help")

## 8. Summary and Recommendations

### Key Takeaways:

1. **max_depth**: Most important for controlling overfitting
   - Start with 10-20 for balanced performance
   - Increase if underfitting, decrease if overfitting

2. **min_samples_split & min_samples_leaf**: Fine-tune generalization
   - Higher values = simpler, more generalizable model
   - min_samples_leaf often more effective than min_samples_split

3. **max_features**: Reduces overfitting and speeds up training
   - 'sqrt' or 'log2' often work well
   - More important for high-dimensional data

4. **criterion**: Usually 'gini' is fine
   - Try 'entropy' if you want more balanced splits

5. **class_weight**: Essential for imbalanced datasets
   - Use 'balanced' to handle class imbalance

### Tuning Strategy:

1. Start with baseline model
2. Adjust max_depth first (biggest impact)
3. Then tune min_samples_split/leaf
4. Finally adjust max_features and criterion
5. Use GridSearchCV for final optimization

### When to Stop Tuning:

- Training and test accuracy are close
- Model generalizes well to new data
- Performance is acceptable for your use case
- Further tuning shows diminishing returns

In [ ]:
# Final model summary
print("="*70)
print("FINAL DECISION TREE MODEL SUMMARY")
print("="*70)
print(f"\nBest Hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nModel Performance:")
print(f"  Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"  Test F1-Score: {f1_score(y_test, y_pred_best, average='weighted'):.4f}")
print(f"  Test Precision: {precision_score(y_test, y_pred_best, average='weighted'):.4f}")
print(f"  Test Recall: {recall_score(y_test, y_pred_best, average='weighted'):.4f}")
cm = confusion_matrix(y_test, y_pred)

print(f"\nModel Complexity:")
print(f"  Tree Depth: {best_dt.get_depth()}")
print(f"  Number of Leaves: {best_dt.get_n_leaves()}")
print(f"  Number of Features: {X_train.shape[1]}")

print(f"\nImprovement over Baseline:")
baseline_acc = accuracy_score(y_test, baseline_dt.predict(X_test))
improvement = ((accuracy_score(y_test, y_pred_best) - baseline_acc) / baseline_acc) * 100
print(f"  Accuracy improvement: {improvement:+.2f}%")
print("="*70)